In [1]:
# ============================================================================
# BLOCKS 2-3: TEXT CLEANING & FILTERING - POLISH
# Input: 01_loaded_data_pl.pkl
# Output: 02_cleaned_data_pl.pkl, 03_filtered_data_pl.pkl
# ============================================================================
%run 00_setup_and_config.ipynb

c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\election_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Libraries loaded successfully
✓ PyTorch device: CPU
✓ Timestamp: 2026-05-29 04:02:29
✓ Directory structure verified
✓ Base directory: c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\YT_Analysis
✓ Model configuration loaded
  - Polish sentiment: eevvgg/bert-polish-sentiment-politics
  - Romanian sentiment: readerbench/ro-sentiment
  - Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  - BERTopic topics: 8
  - Batch size: 8
✓ Checkpoint utility functions loaded
✓ Text cleaning functions loaded
✓ Romanian stopwords: 385 words
✓ Polish stopwords: 511 words
✓ Visualization utility functions loaded
✓ Sentiment prediction function loaded
✓ SETUP AND CONFIGURATION COMPLETE

Ready for analysis. Next steps:
  1. Run Polish pipeline: 01_data_loading_pl.ipynb → 06_engagement_metrics_pl.ipynb
  2. Run Romanian pipeline: 01_data_loading_ro.ipynb → 06_engagement_metrics_ro.ipynb
  3. (Optional) Run comparative analysis: 08_comparative_ana

In [2]:
# CELL 1: Load previous checkpoint
print('='*70)
print('BLOCKS 2-3: TEXT CLEANING & FILTERING - POLISH')
print('='*70)

if check_checkpoint_exists('pl', '03_filtered_data'):
    df_pl = load_checkpoint('pl', '03_filtered_data')
    print('✓ Skipping to filtered data checkpoint')
elif check_checkpoint_exists('pl', '02_cleaned_data'):
    df_pl = load_checkpoint('pl', '02_cleaned_data')
    print('✓ Loading from cleaned data checkpoint')
else:
    df_pl = load_checkpoint('pl', '01_loaded_data')
    if df_pl is None:
        raise FileNotFoundError('Run 01_data_loading_pl.ipynb first')

initial_rows = len(df_pl)
print(f'\nStarting with: {initial_rows:,} rows')

BLOCKS 2-3: TEXT CLEANING & FILTERING - POLISH
✓ Loading checkpoint: 01_loaded_data_pl.pkl

Starting with: 6,891 rows


In [3]:
# CELL 2: Remove null comment_text
print('\n--- BLOCK 2: TEXT CLEANING ---\n')

if 'comment_text' not in df_pl.columns:
    raise KeyError('comment_text column not found')

print('1. Removing rows with null comment_text...')
before = len(df_pl)
df_pl = df_pl.dropna(subset=['comment_text']).copy()
print(f'   Rows removed: {before - len(df_pl):,}')


--- BLOCK 2: TEXT CLEANING ---

1. Removing rows with null comment_text...
   Rows removed: 5


In [4]:
# CELL 3: Clean text
print('\n2. Cleaning text (URLs, emoji, HTML, mentions)...')
df_pl['clean_text'] = df_pl['comment_text'].apply(clean_social_text)


2. Cleaning text (URLs, emoji, HTML, mentions)...


In [5]:
# CELL 4: Remove empty texts
print('\n3. Removing empty or whitespace-only comments...')
df_pl = remove_empty_texts(df_pl, 'clean_text')

# Save checkpoint
save_checkpoint(df_pl, 'pl', '02_cleaned_data')


3. Removing empty or whitespace-only comments...
  Removed 115 empty text rows (1.7%)
✓ Checkpoint saved: 02_cleaned_data_pl.pkl


In [6]:
# CELL 5: Remove duplicates (BLOCK 3)
print('\n--- BLOCK 3: QUALITY FILTERING ---\n')

print('4. Removing exact duplicate comments...')
df_pl = remove_duplicates(df_pl, subset=['video_id', 'author_name', 'comment_text'])


--- BLOCK 3: QUALITY FILTERING ---

4. Removing exact duplicate comments...
  Removed 8 duplicate rows (0.1%)


In [7]:
# CELL 6: Compute text metrics
print('\n5. Computing text statistics...')
df_pl['char_len'] = df_pl['clean_text'].str.len()
df_pl['word_count'] = df_pl['clean_text'].str.split().str.len()

print('\nText Metrics Summary:')
display(df_pl[['char_len', 'word_count']].describe())

# Save final checkpoint
save_checkpoint(df_pl, 'pl', '03_filtered_data')
update_pipeline_status('pl', 2, 'completed')
update_pipeline_status('pl', 3, 'completed')



5. Computing text statistics...

Text Metrics Summary:


,char_len,word_count
count,6763.000000,6763.000000
mean,140.459412,22.011385
std,187.047074,29.262375
min,2.000000,1.000000
25%,47.000000,7.000000
50%,86.000000,14.000000
75%,162.000000,25.500000
max,3256.000000,533.000000


✓ Checkpoint saved: 03_filtered_data_pl.pkl
✓ Pipeline status updated: pl - Block 2 - completed
✓ Pipeline status updated: pl - Block 3 - completed


In [8]:
# CELL 7: Summary
final_rows = len(df_pl)
retention = 100 * final_rows / initial_rows

print('\n' + '='*70)
print('CLEANING SUMMARY - POLISH')
print('='*70)
print(f'  Started with: {initial_rows:,} rows')
print(f'  Ended with:   {final_rows:,} rows')
print(f'  Retention:    {retention:.1f}%')
print(f'  Removed:      {initial_rows - final_rows:,} rows')
print('='*70)
print('✓ BLOCKS 2-3 COMPLETE')
print('='*70)


CLEANING SUMMARY - POLISH
  Started with: 6,891 rows
  Ended with:   6,763 rows
  Retention:    98.1%
  Removed:      128 rows
✓ BLOCKS 2-3 COMPLETE
